# 🧬 bee.Evolver — Colab Runner

Runs the Aura Hive evolutionary cycle on Google Colab's free T4 GPU.
Use this for deeper analysis with larger models when the weekly GitHub Actions cycle isn't enough.

**Before running:**
1. Set your secrets in the *Secrets* panel (🔑 icon in the left sidebar)
2. Required: `MISTRAL_API_KEY`, `GITHUB_TOKEN`, `GITHUB_REPOSITORY`, `AURA_TELEGRAM_TOKEN`, `AURA_BEE_KEEPER__ADMIN_CHAT_ID`
3. Optional: `EVOLVER_FOCUS` — free-text hint (e.g. `"improve bee.Keeper prompts"`)
4. Run all cells in order (Runtime → Run all)

In [ ]:
# Cell 1: Install dependencies
!pip install -q litellm httpx gitpython pydantic-settings structlog

In [ ]:
# Cell 2: Clone the Hive and configure git
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_REPOSITORY = userdata.get('GITHUB_REPOSITORY')  # e.g. 'zaebee/aura'

repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPOSITORY}.git"
repo_dir = "/content/aura"

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}
else:
    !git -C {repo_dir} pull

!git -C {repo_dir} config user.name "bee.Evolver"
!git -C {repo_dir} config user.email "evolver@aura.hive"

# Add agent source to path
import sys
sys.path.insert(0, f"{repo_dir}/agents/bee-evolver/src")
sys.path.insert(0, f"{repo_dir}/agents/bee-evolver")

print(f"Hive cloned to {repo_dir}")

In [ ]:
# Cell 3: Set environment variables from Colab secrets
from google.colab import userdata

def _get_secret(key, default=''):
    try:
        return userdata.get(key) or default
    except Exception:
        return default

os.environ['AURA_LLM__API_KEY']   = _get_secret('MISTRAL_API_KEY')
os.environ['AURA_LLM__MODEL']     = 'mistral/mistral-large-latest'
os.environ['AURA_LLM__FALLBACK_MODEL'] = 'openai/gpt-4o-mini'
os.environ['GITHUB_TOKEN']        = _get_secret('GITHUB_TOKEN')
os.environ['GITHUB_REPOSITORY']   = _get_secret('GITHUB_REPOSITORY')
os.environ['GITHUB_EVENT_NAME']   = 'manual'
os.environ['AURA_TELEGRAM_TOKEN'] = _get_secret('AURA_TELEGRAM_TOKEN')
os.environ['AURA_BEE_KEEPER__ADMIN_CHAT_ID'] = _get_secret('AURA_BEE_KEEPER__ADMIN_CHAT_ID', '0')
os.environ['EVOLVER_FOCUS']       = _get_secret('EVOLVER_FOCUS', '')
os.environ['EVOLVER_MAX_IMPROVEMENTS'] = '5'   # More improvements on Colab
os.environ['AURA_BEE_EVOLVER__MAX_TOKENS'] = '4000'  # Larger budget on Colab
os.environ['EVOLVER_ASSIGNEE']    = 'zaebee'

print("Environment configured.")
print(f"  Model:      {os.environ['AURA_LLM__MODEL']}")
print(f"  Repository: {os.environ['GITHUB_REPOSITORY']}")
print(f"  Focus:      {os.environ.get('EVOLVER_FOCUS') or '(none)'}")

In [ ]:
# Cell 4: Run Aggregator — sense the Hive
import asyncio
import json

# Patch find_hive_root to point at the cloned repo
import aura_core
from pathlib import Path
aura_core._HIVE_ROOT = Path(repo_dir)  # type: ignore

from config import EvolverSettings
from hive.aggregator import EvolverAggregator

settings = EvolverSettings()
aggregator = EvolverAggregator(settings)
context = asyncio.get_event_loop().run_until_complete(aggregator.perceive())

print(f"Git log lines:    {len(context.git_log.splitlines())}")
print(f"Open issues:      {len(context.open_issues)}")
print(f"Open PRs:         {len(context.open_prs)}")
print(f"Recent heresies:  {len(context.recent_heresies)}")
print(f"Focus hint:       {context.focus_hint or '(none)'}")

In [ ]:
# Cell 5: Run Transformer — generate EvolutionPlan
from hive.transformer import EvolverTransformer

transformer = EvolverTransformer(settings)
plan = asyncio.get_event_loop().run_until_complete(transformer.think(context))

print(f"Improvements:  {len(plan.improvements)}")
print(f"Tokens used:   {plan.token_usage}")
print(f"Optimal:       {plan.hive_is_optimal}")
print(f"Narrative:     {plan.narrative}")
print()
for i, imp in enumerate(plan.improvements, 1):
    print(f"{i}. [{imp.type}] {imp.title}")
    print(f"   {imp.description}")
    print(f"   target_file: {imp.target_file}")
    print()

In [ ]:
# Cell 6: Inspect patches before applying (optional review step)
for imp in plan.improvements:
    if imp.patch:
        print(f"=== [{imp.type}] {imp.title} ===")
        print(f"Target: {imp.target_file}")
        print(imp.patch[:1000])
        print()

In [ ]:
# Cell 7: Apply patches, push branch, open PR + Issues, send Telegram pulse
# ⚠️  This cell makes real changes to the repository. Review Cell 6 first.
from hive.metabolism import EvolverMetabolism

metabolism = EvolverMetabolism(settings)
# Inject the already-computed plan to skip re-running the LLM
metabolism._precomputed_plan = plan  # type: ignore

observation = asyncio.get_event_loop().run_until_complete(metabolism.execute())

print(f"Success:        {observation.success}")
print(f"PR URL:         {observation.pr_url or '(none)'}")
print(f"Issues created: {len(observation.issue_urls)}")
print(f"Telegram sent:  {observation.telegram_sent}")
if observation.errors:
    print(f"Errors:")
    for e in observation.errors:
        print(f"  - {e}")

## Done

If a PR was opened, review it at the URL above before merging.
All commits include `[skip ci]` to prevent CI loops.

_🐝 For the glory of the Hive._